# Продажи в черную пятницу

Данные о покупках в розничном магазине во время "Черной пятницы".

Цель: Понять профиль «крупного покупателя» и выяснить, какие категории товаров приносят больше всего выручки.

[ссылка на датасет](https://www.kaggle.com/datasets/pranavuikey/black-friday-sales-eda)

Качаем датасет

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("pranavuikey/black-friday-sales-eda")

print("Путь к датасету", path)

Грузим датасет

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sb
df = pd.read_csv(path + "/train.csv")

### Быстрый обзор

Первые пять записей

In [ ]:
df.head()

Разрешение фрейма

In [ ]:
df.shape

Краткая информация о фрейме

In [ ]:
df.describe()

In [ ]:
df.describe(include='object')

In [ ]:
df.info()

### Проверяем данные и заполняем пропуски

Ищем пропуски:

In [ ]:
df.isnull().sum()

Категории 2 и 3 содержат пропуски, однако это не ошибки, а просто у товара нет соответствующей категории. Заполняем 0.

In [ ]:
df = df.fillna(0)

Проверим дубликаты

In [ ]:
df.duplicated().sum()

Дубликатов нет

In [ ]:
df.median(numeric_only=True)

Видим, что типичная покупка составляет сумму 8047. Пусть будет долларов.

In [ ]:
df.mean(numeric_only=True)

Среднее значение больше чем медиана, что значит, что наблюдается хвост справа, то есть совершалось большое количество аномально больших покупок.

In [ ]:
df.min()

In [ ]:
df.max()

In [ ]:
df.mode()

Отсюда видим, что чаще всего совершают покупки неженатые мужчины 26-35 лет, живущие 1 год в городах типа B.

In [ ]:
df.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95])

In [ ]:
# Дисперсия
df.var(numeric_only=True)

У нас большой разброс по цене

In [ ]:
# Асимметрия
df.skew(numeric_only=True)

Видим, что асимметрия положительная, что говорит о "длинном хвосте" справа. Значит, у нас есть некоторые покупатели, которые закупаются на высокие суммы.

In [ ]:
# Эксцес
df.kurt(numeric_only=True)

Отрицательный экцес (платикуртозис) говорит о том, что у нас не очень много выбросов и клиенты более-менее равномерно совершают покупки.

### Фичи: Энкодинг и Ижиниринг

По идее, сумма покупки должна зависить от стоимости категории продукта.

In [ ]:
df['Category_Avg_Purchase'] = df.groupby('Product_Category_1')['Purchase'].transform('mean')

Сделаем признак, который будет отражать частоту покупки некой категории в данном городе.

In [ ]:
df['Category_City_Count'] = df.groupby(['City_Category', 'Product_Category_1'])['User_ID'].transform('count')

Сделаем энкодинг признаков Gender и City_category, тк в них четко прослеживаются независимые категории

In [ ]:
cols_enc = ['Gender', 'City_Category']

gender_city_enc_dummies = pd.get_dummies(df[cols_enc], drop_first=True)
df_enc = pd.concat([df, gender_city_enc_dummies], axis=1)
df_enc.drop(cols_enc, axis=1, inplace=True)

In [ ]:
df_enc

Для признака Stay_In_Current_City_Years нам нужно использовать OrdinalEncoder, тк нам важно, что 1 < 2, 0 < 4 и тп.

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

age_order = ['0-17', '18-25', '26-35', '36-45', '46-50', '51-55', '55+']
stay_order = ['0', '1', '2', '3', '4+']

encoder = OrdinalEncoder(categories=[age_order, stay_order])
df_enc[['Age', 'Stay_In_Current_City_Years']] = encoder.fit_transform(
    df_enc[['Age', 'Stay_In_Current_City_Years']]
)

In [ ]:
df_enc

Поскольку у нас нет дублированных User_ID, это значит, что мы не сможем узнать, закупается ли пользователь на сооветствующую сумму постоянно или только один раз, т.е. мы не сможем определить китов. Следовательно, и User_ID нам не нужен. То же самое с Product_ID.

In [ ]:
df_enc = df_enc.drop(['User_ID', 'Product_ID'], axis=1)

### Визуализация

Давайте посмотрим на гистограмму.

In [ ]:
import seaborn as sb
import matplotlib.pyplot as plt

plt.figure(figsize=(15,8))
sb.histplot(df['Purchase'], kde=True, color='blue')
plt.title('Распределение сумм покупок')
plt.show()

Из нее видно, что больше всего покупок совершается на сумму в интервале от 5000 до 10000. Так же справа мы видим тот самый длинный хвост.

Мы можем убрать этот хвост, если возьмем логарифм. Например, если у нас есть покупки на суммы 10, 1000, 100000, то в обычной системе счисления разница очень высока, что увеличивает разброс предсказаний на более дорогих покупках. Если возьмем логарифм (пример, основание 10), то покупки будут стоить 1, 3, 5. Тут уже разница небольшая и алгоритму будет легче подобрать гиперплоскость (случай линейной регрессии или svm). **!! это писал не ии !!**

In [ ]:
df2 = df.copy()
df2.Purchase = np.log1p(df2.Purchase) # 1p - плюс 1, чтобы не было логарифма от 0.
sb.histplot(df2['Purchase'], kde=True, color='blue')

Как видим, структура стала более предсказуемой.

Теперь попробуем определить зависимости с помощью scatterplot.

In [ ]:
import plotly.express as px

fig = px.scatter(df.sample(2000), x="Occupation", y="Purchase", color="Gender",
                 hover_data=['Age', 'City_Category'], 
                 title="Зависимость суммы покупки от профессии и пола")
fig.show()


Заметим, что меньше всего покупок совершают люди с профессией 8.
Еще заметим, что профессией 18 заняты только мужчины ( в выборке 2000 )

Поищем аномалии с помощью boxplot.

In [ ]:
plt.figure(figsize=(15, 8))
plt.title('Анализ выбросов по возрастным категориям')
sb.boxplot(x='Age', y='Purchase', hue='Gender', data=df)
plt.show()

Как видим, медианные стоимости примерно одинаковы для всех возрастных категорий. 
Мужчины систематически тратят больше женщин по медиане.
Немного странно, что подростки тратят столько же, сколько, например, люди от 18 до 25, возможно возраст слабый предиктор для данного датасета.
Сверху у нас много выбросов. Это всё мажоры.

In [ ]:
plt.figure(figsize=(15, 8))
plt.title('Анализ выбросов по категориям продуктов')
sb.boxplot(x='Product_Category_1', y='Purchase', hue='Gender', data=df)
plt.show()

Как видим, тут медианы разбросаны довольно сильно. Из-за того что все остальные признаки особо не отличаются, этот признак является самым информативным.

Проверим, сколько в каких городах делают покупок.

In [ ]:
order = ['A','B','C']

plt.figure(figsize=(15, 8))
sb.countplot(x='City_Category', order=order, data=df)
plt.title('Количество покупок по категориям городов')
plt.show()

Как видим, в городах типа B их делают больше всего. Возможно города B - это мегаполисы. Поскольку мы убедились, что все покупатели уникальны, то получается, что в городе B просто больше людей.

Построим heatmap корреляций

In [ ]:
plt.figure(figsize=(10, 8))
corr_matrix = df_enc.select_dtypes(include=['float64', 'int64']).corr()
sb.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Тепловая карта корреляций')
plt.show()

Можем заметить, что у нас появилась проблема мультиколлиннеарности из-за category_avg_purchase и category_city_count, тк они несут похожие функции с purchase.
Больше всего с Purchase коррелируют признаки Product_Category_*
Еще можем заметить, что возрастная категория влияет на семейное положение - что логично.

### Выводы

1. Исходя из датасета, мужчины тратят чуть больше женщин.
2. Люди с профессией 8 покупают меньше всего.
3. Больше всего людей тратят от 5 до 10 тысяч, однако есть "киты", которые тратят больше.
4. Люди в городах категории B совершают больше всего покупок.
5. Чаще всего совершают покупки неженатые мужчины 26-35 лет, живущие 1 год в городах типа B.
6. Типичная покупка составляет 8047 долларов.
7. Самая большая и маленькая покупки составляют 23961 и 12 долларов соответственно.

### Гипотезы
* Категория продукта больше всего влияет на стоимость.
* В первый год люди тратят больше, тк нужно обустраивать свой дом
* Профессия 8 самая информативная (тк если у человека она, то вероятно, он потратит меньше)
* У нас есть богачи, которые тратят аномально много и генерируют выбросы.

### Что дальше?

* Я бы использовал линейную регрессию как базу, относительно которой можно понимать, насколько хороши остальные модели
* С помощью RandomForestsRegressor можно проверить мои гипотезы про важность фич

### Что я просил у ИИ?
Я попросил составить план eda для данного датасета у gemini (notebooklm с загруженным яндекс хэндбуком и другими источниками), и объяснить, зачем мне делать каждый шаг и какую информацию он мне даёт.

И еще спросил, как лучше закодировать возраст, ведь там важна иерархия.

Остальное писал сам :)